# Trabalho de APM - Regras de Associação - G13

### 1. Importar bibliotecas

In [1]:
import pandas as pd

from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings("ignore")


### 2. Macro definições

In [2]:
# Bases de dados
DF_MUSCULACAO = "Musculacao - Dados.csv"

# Minimo support
MIN_SUP = 0.3 

# Minimo Threshold
MIN_THR = 0.8

# Largura máxima das colunas a serem exibidas
pd.set_option('display.max_colwidth', None)

### 3. Funções locais

In [3]:
# Carrega csv com tamanho variável de itens por linha
import csv

transacoes = []

with open(DF_MUSCULACAO, encoding='utf-8') as f:
    leitor = csv.reader(f, delimiter=';')
    for linha in leitor:
        # remove campos vazios
        itens = [item.strip() for item in linha if item.strip()]
        transacoes.append(itens)

print("Amostra dos dados:")
for transacao in transacoes[0:5]:
    print (transacao)

Amostra dos dados:
['LegPress', 'Gemeos', 'Afundo', 'Crucifixo']
['LegPress', 'Gemeos', 'Agachamento']
['Gemeos', 'LegPress', 'Afundo', 'Agachamento']
['Adutor', 'Agachamento', 'LegPress', 'Adutor']
['LegPress', 'Gemeos', 'Afundo', 'Bicicleta']


### 4. Cria as regras de associação

In [4]:
# -----------------------------
# 1. One-hot encoding
# -----------------------------
te = TransactionEncoder()
te_array = te.fit(transacoes).transform(transacoes)

df = pd.DataFrame(te_array, columns=te.columns_) # Base binarias das transações

# -----------------------------
# 2. Itemsets frequentes
# -----------------------------
itemsets = apriori(
    df,
    min_support=MIN_SUP,
    use_colnames=True
)

# lista apenas os itens (remove o 'frozenset')
itemsets['itens_str'] = itemsets['itemsets'].apply(
    lambda x: ', '.join(sorted(x))
)

print('\nItemsets frequentes:\n')
print(itemsets[['support', 'itens_str']].sort_values(by='support', ascending=False))


Itemsets frequentes:

     support                              itens_str
7   0.807692                               LegPress
6   0.653846                                 Gemeos
3   0.538462                              Bicicleta
5   0.500000                               Extensor
4   0.461538                                Esteira
14  0.461538                    Bicicleta, Extensor
20  0.461538                       Gemeos, LegPress
17  0.423077                      Esteira, Extensor
13  0.384615                     Bicicleta, Esteira
2   0.384615                       AgachamentoSmith
22  0.384615           Bicicleta, Esteira, Extensor
16  0.346154                    Bicicleta, LegPress
0   0.346154                                 Afundo
12  0.346154             AgachamentoSmith, Extensor
8   0.346154                         Afundo, Gemeos
1   0.307692                            Agachamento
11  0.307692              AgachamentoSmith, Esteira
9   0.307692                  Agachamento

In [5]:
# -----------------------------
# 3. Regras de associação
# -----------------------------
regras = association_rules(
    itemsets,
    metric='confidence',
    min_threshold=MIN_THR
)

# Seleciona colunas mais importantes
regras = regras[[
    'antecedents',
    'consequents',
    'support',
    'confidence',
    'lift'
]]

regras['antecedents_str'] = regras['antecedents'].apply(
    lambda x: ', '.join(sorted(x))
)

regras['consequents_str'] = regras['consequents'].apply(
    lambda x: ', '.join(sorted(x))
)
print('Regras encontradas:')
print(regras[['antecedents_str', 'consequents_str', 'confidence', 'support', 'lift']]
        .sort_values(by='confidence', ascending=False)
        .to_string(index=False)
)

Regras encontradas:
            antecedents_str     consequents_str  confidence  support     lift
                     Afundo              Gemeos    1.000000 0.346154 1.529412
                Agachamento            LegPress    1.000000 0.307692 1.238095
AgachamentoSmith, Bicicleta            Extensor    1.000000 0.307692 2.000000
         Bicicleta, Esteira            Extensor    1.000000 0.384615 2.000000
                   Extensor           Bicicleta    0.923077 0.461538 1.714286
                    Esteira            Extensor    0.916667 0.423077 1.833333
          Esteira, Extensor           Bicicleta    0.909091 0.384615 1.688312
           AgachamentoSmith            Extensor    0.900000 0.346154 1.800000
 AgachamentoSmith, Extensor           Bicicleta    0.888889 0.307692 1.650794
                  Bicicleta            Extensor    0.857143 0.461538 1.714286
                   Extensor             Esteira    0.846154 0.423077 1.833333
                    Esteira           Bicicl